# Desenvolvimento do Modelo
### O mercado de shows em São Paulo (2021–2025)

##### O projeto pretende entender o comportamento e fluxo de público em shows médio e grande em São Paulo, considerando a popularidade do nome no spotify, o genero qdo nome, o local onde ocorrem os shows, valores de ingresso e data.

In [1]:
%matplotlib inline

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patheffects as patheffects
import seaborn as sns

from phik.report import plot_correlation_matrix

import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn import metrics
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

from pycaret.regression import *

from ydata_profiling import ProfileReport

import pickle

In [2]:
df = pd.read_csv('../data/processed/base_modelo.csv', sep = ',')
df.head()

,ano,popularidade,lotacao,tipo_dia,mes,distancia_dias_anterior,genero_cluster
0,2023,79.0,8000.0,DIA UTIL,JULHO,-11.500000,14.0
1,2022,69.0,3200.0,DIA UTIL,MARÇO,-81.500000,0.0
2,2022,72.0,8000.0,DIA UTIL,JULHO,-167.666667,Outros
3,2022,72.0,8000.0,DIA UTIL,JULHO,0.333333,Outros
4,2025,76.0,8000.0,DIA UTIL,AGOSTO,333.000000,Outros


In [3]:
ProfileReport(df, title='Profiling Report da Base de Trabalho')

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|███████████████████████████████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 115.10it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Para separação dos dados de teste e treino, considerando as 1159 entradas, utilizarei para teste, aproximadamente, 20% de registros para teste. 

### Utilizando Pycaret

Utilizando para definição de um modelo inicial

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 848 entries, 0 to 847
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ano                      848 non-null    int64  
 1   popularidade             848 non-null    float64
 2   lotacao                  848 non-null    float64
 3   tipo_dia                 848 non-null    object 
 4   mes                      848 non-null    object 
 5   distancia_dias_anterior  848 non-null    float64
 6   genero_cluster           848 non-null    object 
dtypes: float64(3), int64(1), object(3)
memory usage: 46.5+ KB


In [5]:
cat = ['genero_cluster', 'tipo_dia', 'mes']
exp = setup(data = df, 
            target = 'lotacao',
            session_id = 42,
            normalize = True,           
            categorical_features = cat,  
            train_size = 0.8)

best = compare_models(fold=10, sort='R2') 

tuned = tune_model(best)

py_bst = finalize_model(tuned)

,Description,Value
0,Session id,42
1,Target,lotacao
2,Target type,Regression
3,Original data shape,"(848, 7)"
4,Transformed data shape,"(848, 36)"
5,Transformed train set shape,"(678, 36)"
6,Transformed test set shape,"(170, 36)"
7,Numeric features,3
8,Categorical features,3
9,Preprocess,True


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
rf,Random Forest Regressor,10799.9518,232536872.1240,15157.6788,0.4204,1.4344,1.2614,0.0470
gbr,Gradient Boosting Regressor,11290.3916,239579735.5347,15351.0404,0.4083,1.4497,1.2833,0.0250
et,Extra Trees Regressor,9847.6170,240121266.3017,15400.1689,0.4001,1.4012,1.2025,0.0390
br,Bayesian Ridge,12276.1594,266830429.4722,16247.3049,0.3358,1.5265,1.4190,0.0170
en,Elastic Net,12652.9070,270512104.0535,16345.1764,0.3299,1.4990,1.4768,0.0180
ridge,Ridge Regression,12206.5410,268961053.0549,16319.3089,0.3288,1.5542,1.4054,0.0190
lar,Least Angle Regression,12205.9838,269012436.4713,16320.9773,0.3286,1.5566,1.4053,0.0180
llar,Lasso Least Angle Regression,12205.9295,268999837.6214,16320.5594,0.3286,1.5556,1.4053,0.0170
lasso,Lasso Regression,12205.9821,268999033.2415,16320.5307,0.3286,1.5558,1.4053,0.3040
lr,Linear Regression,12191.1623,269295258.8531,16329.9512,0.3275,1.5249,1.3962,0.3120


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,12908.6271,295602342.9859,17193.0900,0.3647,1.0133,1.6231
1,12584.5930,254462167.1123,15951.8703,0.3178,0.8282,1.0016
2,9354.2994,152385393.3710,12344.4479,0.4500,0.9225,1.4122
3,14660.7017,362560634.5812,19041.0250,0.3447,2.2196,1.2174
4,12189.9788,238578950.7660,15446.0011,0.2143,1.9294,1.4090
5,11788.4509,254922646.6317,15966.2972,0.5010,1.4177,1.3505
6,12225.0043,273098552.0553,16525.6937,0.3855,1.4938,1.5465
7,13309.4291,273145393.6051,16527.1109,0.3458,1.8880,1.5630
8,10245.1513,170689195.2579,13064.8075,0.4423,1.4804,1.0609


Fitting 10 folds for each of 10 candidates, totalling 100 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


### Dividindo teste e treino

In [6]:
# Variável resposta
y = df['lotacao']

# Variaveis dependentes
X = df.drop(['lotacao'], axis = 1)

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Modelagem linear

Considerando os dados e relacionamentos anteriormente encontrados, os modelos de suposições.

In [8]:
train_df = X_train.copy()
train_df['lotacao'] = y_train

test_df = X_test.copy()
test_df['lotacao'] = y_test

In [9]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 678 entries, 598 to 102
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ano                      678 non-null    int64  
 1   popularidade             678 non-null    float64
 2   tipo_dia                 678 non-null    object 
 3   mes                      678 non-null    object 
 4   distancia_dias_anterior  678 non-null    float64
 5   genero_cluster           678 non-null    object 
 6   lotacao                  678 non-null    float64
dtypes: float64(3), int64(1), object(3)
memory usage: 42.4+ KB


In [10]:
formula_1 = '''
    lotacao ~ ano + C(genero_cluster) + I(popularidade**0.5) + C(tipo_dia) + C(mes) + distancia_dias_anterior
'''
md1 = smf.glm(formula_1, data=train_df, family=sm.families.Gaussian()).fit()  

print(md1.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                lotacao   No. Observations:                  678
Model:                            GLM   Df Residuals:                      645
Model Family:                Gaussian   Df Model:                           32
Link Function:               Identity   Scale:                      2.5545e+08
Method:                          IRLS   Log-Likelihood:                -7507.7
Date:                Sun, 18 Jan 2026   Deviance:                   1.6477e+11
Time:                        22:35:05   Pearson chi2:                 1.65e+11
No. Iterations:                     3   Pseudo R-squ. (CS):             0.4843
Covariance Type:            nonrobust                                         
                                       coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------------
Intercep

In [11]:
pred_test = md1.predict(test_df)
print(f"MAE: {metrics.mean_absolute_error(y_test, pred_test):.2f}")
print(f"R²: {metrics.r2_score(y_test, pred_test):.3f}")

MAE: 12831.05
R²: 0.366


Em comparação ao modelo de Random Forest selecionado pelo PyCaret, este modelo apresentou ganhos nos indicadores Pseudo-R² e R². Apesar da evolução observada, o processo de modelagem seguirá em busca de melhorias adicionais.

In [12]:
formula_2 = '''
      lotacao ~  I(popularidade**2) + C(genero_cluster) + C(tipo_dia) * C(mes) * ano + distancia_dias_anterior
'''
md2= smf.glm(formula_2, data=train_df, family=sm.families.Gaussian()).fit()  

print(md2.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                lotacao   No. Observations:                  678
Model:                            GLM   Df Residuals:                      603
Model Family:                Gaussian   Df Model:                           74
Link Function:               Identity   Scale:                      2.4212e+08
Method:                          IRLS   Log-Likelihood:                -7466.7
Date:                Sun, 18 Jan 2026   Deviance:                   1.4600e+11
Time:                        22:35:05   Pearson chi2:                 1.46e+11
No. Iterations:                     3   Pseudo R-squ. (CS):             0.5589
Covariance Type:            nonrobust                                         
                                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [13]:
pred_test = md2.predict(test_df)
print(f"MAE: {metrics.mean_absolute_error(y_test, pred_test):.2f}")
print(f"R²: {metrics.r2_score(y_test, pred_test):.3f}")

MAE: 12198.88
R²: 0.408


Houve uma grande melhora no valor de Pseudo-R² e uma melhora no valor de R². A próxima etapa consiste em testar diferentes interações entre as variáveis, buscando aprimorar ainda mais o desempenho do modelo.

In [14]:
formula_3 = '''
      lotacao ~  I(popularidade**2) * C(genero_cluster) + C(tipo_dia) * C(mes) * ano + distancia_dias_anterior
'''
md3= smf.glm(formula_3, data=train_df, family=sm.families.Gaussian()).fit()  

print(md3.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                lotacao   No. Observations:                  678
Model:                            GLM   Df Residuals:                      588
Model Family:                Gaussian   Df Model:                           89
Link Function:               Identity   Scale:                      2.4143e+08
Method:                          IRLS   Log-Likelihood:                -7457.2
Date:                Sun, 18 Jan 2026   Deviance:                   1.4196e+11
Time:                        22:35:05   Pearson chi2:                 1.42e+11
No. Iterations:                     3   Pseudo R-squ. (CS):             0.5719
Covariance Type:            nonrobust                                         
                                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [15]:
pred_test = md3.predict(test_df)
print(f"MAE: {metrics.mean_absolute_error(y_test, pred_test):.2f}")
print(f"R²: {metrics.r2_score(y_test, pred_test):.3f}")

MAE: 12082.23
R²: 0.407


In [16]:
formula_4= '''
      lotacao ~  I(popularidade**2) * C(genero_cluster) + C(tipo_dia) * C(mes) * ano + I(distancia_dias_anterior) * distancia_dias_anterior
'''
md4= smf.glm(formula_4, data=train_df, family=sm.families.Gaussian()).fit()  

print(md4.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                lotacao   No. Observations:                  678
Model:                            GLM   Df Residuals:                      587
Model Family:                Gaussian   Df Model:                           90
Link Function:               Identity   Scale:                      2.4114e+08
Method:                          IRLS   Log-Likelihood:                -7456.2
Date:                Sun, 18 Jan 2026   Deviance:                   1.4155e+11
Time:                        22:35:05   Pearson chi2:                 1.42e+11
No. Iterations:                     3   Pseudo R-squ. (CS):             0.5735
Covariance Type:            nonrobust                                         
                                                               coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------

In [17]:
pred_test = md4.predict(test_df)
print(f"MAE: {metrics.mean_absolute_error(y_test, pred_test):.2f}")
print(f"R²: {metrics.r2_score(y_test, pred_test):.3f}")

MAE: 12035.61
R²: 0.407


In [18]:
formula_5 = '''
      lotacao ~ I(popularidade**2) * C(genero_cluster) + C(mes) * I(ano**2) +  I(distancia_dias_anterior) * distancia_dias_anterior
'''
md5= smf.glm(formula_5, data=train_df, family=sm.families.Gamma(link=sm.families.links.log())).fit()  

print(md5.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                lotacao   No. Observations:                  678
Model:                            GLM   Df Residuals:                      621
Model Family:                   Gamma   Df Model:                           56
Link Function:                    log   Scale:                         0.76877
Method:                          IRLS   Log-Likelihood:                    inf
Date:                Sun, 18 Jan 2026   Deviance:                       1204.0
Time:                        22:35:06   Pearson chi2:                     477.
No. Iterations:                   100   Pseudo R-squ. (CS):                nan
Covariance Type:            nonrobust                                         
                                                         coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------

In [19]:
pred_test = md5.predict(test_df)
print(f"MAE: {metrics.mean_absolute_error(y_test, pred_test):.2f}")
print(f"R²: {metrics.r2_score(y_test, pred_test):.3f}")

MAE: 13189.93
R²: 0.270


Este último modelo está usando um conceito Gamma e não Gaussiano como os anteriores. Nessa base o Pseudo R² entre 0,2 e 0,4 já tem boa aceitabilidade.

Após a análise das métricas de desempenho (R², MAE, RMSE, entre outras), observou-se que o modelo que apresentou os melhores resultados foi o **quinto modelo desenvolvido e testado**.

In [20]:
X_sample = df.drop(columns=['lotacao']).copy()

try:
    exog_sample = X_sample.head(5)  
    pred_sample = md5.predict(exog=exog_sample)
    print("✅ Validação OK! Previsão de exemplo:")
    print(pred_sample)
    print("Valores previstos:", pred_sample.values.round(0))

except Exception as e:
    print("❌ Erro na validação do modelo:")
    print(e)
    raise

✅ Validação OK! Previsão de exemplo:
0    21117.613984
1     9379.554274
2     8171.568827
3     7711.397886
4    17869.258950
dtype: float64
Valores previstos: [21118.  9380.  8172.  7711. 17869.]


In [21]:
with open('../data/models/modelo_final.pkl', 'wb') as file:
    pickle.dump(md5, file)

In [22]:
with open('../data/models/reference_columns.pkl', 'wb') as file:
    pickle.dump(X_sample.columns.tolist(), file)